# 22 · Managed vector search over Delta data

**Prerequisites:** 00; writable Unity Catalog schema; existing search and embedding endpoints; appropriate permissions/quota.

**Execution:** optional. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Create a Delta Sync index optionally; understand endpoint permissions, embeddings and synchronization.


## Step 1 — Inspect required configuration

This advanced lab uses an existing STANDARD AI Search/Vector Search endpoint and an existing embedding model endpoint. Install `databricks-vectorsearch>=0.50,<1` in the notebook environment. No resources are created by default.


In [ ]:
RUN_MANAGED_SEARCH=False
SEARCH_ENDPOINT=''
EMBEDDING_ENDPOINT=''
INDEX_NAME=''  # catalog.schema.unique_course_index
SOURCE_TABLE='' # catalog.schema.unique_course_documents
print('Managed search enabled:',RUN_MANAGED_SEARCH)


**Check your result:** Default mode is a dry run.


## Step 2 — Create a governed source table

Choose course-only names in a writable schema. This lab creates the table if absent and MERGEs the six sample documents. A Delta Sync source needs a primary key and change data feed on standard endpoints.


In [ ]:
documents = [
    {'id':'delta', 'text':'Delta Lake supports ACID transactions and table version history.'},
    {'id':'catalog', 'text':'Unity Catalog controls permissions and governs tables and models.'},
    {'id':'mlflow', 'text':'MLflow tracks experiments, parameters, metrics, and model artifacts.'},
    {'id':'spark', 'text':'Spark processes large datasets using distributed DataFrame operations.'},
    {'id':'rag', 'text':'Retrieval augmented generation retrieves evidence before generating an answer.'},
    {'id':'batch', 'text':'Batch inference scores many records together on a schedule.'},
]

if RUN_MANAGED_SEARCH:
    import re
    from pyspark.sql import functions as F
    for name in [INDEX_NAME,SOURCE_TABLE]:
        assert re.fullmatch(r'[A-Za-z_][\w]*\.[A-Za-z_][\w]*\.[A-Za-z_][\w]*',name), 'Use simple three-part course names'
    assert SEARCH_ENDPOINT and EMBEDDING_ENDPOINT
    source=spark.createDataFrame([(d['id'],d['text']) for d in documents],'id string, text string')
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SOURCE_TABLE} (id STRING, text STRING) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    source.createOrReplaceTempView('ai_search_docs_source')
    spark.sql(f"MERGE INTO {SOURCE_TABLE} t USING ai_search_docs_source s ON t.id=s.id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
else:
    print('Source preparation skipped')


**Check your result:** Enabled mode prepares only the explicitly named source table.


## Step 3 — Create or reuse a Delta Sync index

The SDK handles managed embeddings. Index creation and initial embedding can consume quota. Existing indexes must match these source/embedding settings; inspect them in the UI before reuse.


In [ ]:
if RUN_MANAGED_SEARCH:
    from databricks.vector_search.client import VectorSearchClient
    vsc=VectorSearchClient()
    listed=vsc.list_indexes(name=SEARCH_ENDPOINT).get('vector_indexes',[])
    if any(item['name']==INDEX_NAME for item in listed):
        index=vsc.get_index(endpoint_name=SEARCH_ENDPOINT,index_name=INDEX_NAME)
    else:
        index=vsc.create_delta_sync_index(endpoint_name=SEARCH_ENDPOINT,index_name=INDEX_NAME,
            source_table_name=SOURCE_TABLE,pipeline_type='TRIGGERED',primary_key='id',
            embedding_source_column='text',embedding_model_endpoint_name=EMBEDDING_ENDPOINT)
    from datetime import timedelta
    index.wait_until_ready(timeout=timedelta(minutes=10))
    index.sync()
    index.wait_until_ready(timeout=timedelta(minutes=10),wait_for_updates=True)
    print(index.describe())
else:
    print('Index creation/sync skipped')


**Check your result:** Enabled mode waits for the initial index and a triggered update.


## Step 4 — Search for evidence

The query uses the managed embedding model. Inspect returned IDs, text, and scores before inserting evidence into a RAG prompt.


In [ ]:
if RUN_MANAGED_SEARCH:
    result=index.similarity_search(query_text='Which tool tracks experiments?',columns=['id','text'],num_results=2)
    print(result)
else:
    print('Use notebook 19 or 21 for a local retrieval lab without managed services.')


**Check your result:** Enabled results should include the MLflow source document.


## Practice

Update one document, trigger sync, and confirm the new content appears. Delete only your course index through the UI after finishing.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Delta source changes are visible after synchronization. A shared search endpoint may have other indexes; do not delete it to clean up one course index.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
